# Notebook 03 — Exploratory Data Analysis

**Objective:** Understand the data through visualisations, identify patterns, and connect EDA insights to modelling decisions.

**Minimum 3 visualisations required for top marks.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.utils import IPC_COLOURS, SEASON_COLOURS, section

os.makedirs("reports/figures", exist_ok=True)

# Load processed data
master   = pd.read_csv("data/processed/master_dataset.csv")
nasa     = pd.read_csv("data/processed/nasa_monthly_clean.csv")
ipc      = pd.read_csv("data/processed/ipc_clean.csv")
knbs     = pd.read_csv("data/processed/knbs_cpi_structured.csv")
news     = pd.read_csv("data/processed/news_clean.csv")

print("Master shape:", master.shape)

## Visualisation 1 — Rainfall Distribution Across Counties

In [ ]:
section("VIZ 1 — Monthly Rainfall by County")

# Average monthly rainfall per county
county_rain = nasa.groupby("county")["total_rainfall"].mean().sort_values()

fig, ax = plt.subplots(figsize=(12, 8))
county_rain.plot(kind="barh", ax=ax, color="#1565C0", alpha=0.8)
ax.set_title("Average Monthly Rainfall by County (2000–2023)", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean Monthly Rainfall (mm)")
ax.axvline(county_rain.mean(), color="red", linestyle="--", label=f"Mean: {county_rain.mean():.1f}mm")
ax.legend()
plt.tight_layout()
plt.savefig("reports/figures/01_rainfall_by_county.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/figures/01_rainfall_by_county.png")

## Visualisation 2 — IPC Food Security Phase Distribution

In [ ]:
section("VIZ 2 — IPC Phase Distribution Across Counties")

ipc_counts = ipc.groupby(["county", "ipc_phase"]).size().reset_index(name="count")
phase_labels = {1: "Minimal", 2: "Stressed", 3: "Crisis"}
ipc_counts["phase_label"] = ipc_counts["ipc_phase"].map(phase_labels)

fig, ax = plt.subplots(figsize=(10, 6))
pivot = ipc_counts.pivot(index="county", columns="phase_label", values="count").fillna(0)
pivot.plot(kind="bar", ax=ax, color=["#66BB6A", "#FFA726", "#EF5350"], stacked=False)
ax.set_title("IPC Food Security Phase Distribution by County (March 2026)", fontsize=13, fontweight="bold")
ax.set_xlabel("County")
ax.set_ylabel("Number of Livelihood Zones")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.savefig("reports/figures/02_ipc_phase_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Visualisation 3 — KNBS CPI Food Price Trends

In [ ]:
section("VIZ 3 — KNBS CPI Food Index Over Time")

if "date" in knbs.columns and "overall_cpi" in knbs.columns:
    knbs["date"] = pd.to_datetime(knbs["date"])
    knbs_valid = knbs.dropna(subset=["overall_cpi"]).sort_values("date")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(knbs_valid["date"], knbs_valid["overall_cpi"],
            color="#2E7D32", linewidth=2, marker="o", markersize=4)
    ax.fill_between(knbs_valid["date"], knbs_valid["overall_cpi"],
                    alpha=0.15, color="#2E7D32")
    ax.set_title("Kenya Overall CPI Over Time (KNBS, 2021–2025)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Consumer Price Index (Base: Feb 2019=100)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("reports/figures/03_knbs_cpi_trend.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("CPI columns not yet extracted — run notebook 02 first")

## Visualisation 4 — Rainfall vs IPC Phase Correlation

In [ ]:
section("VIZ 4 — NASA Rainfall Anomaly vs IPC Phase")

# Join monthly rainfall anomaly with IPC phase
if "ipc_phase_county" in master.columns:
    ipc_weather = master.dropna(subset=["ipc_phase_county", "spi_3"])

    fig, ax = plt.subplots(figsize=(8, 5))
    phase_labels = {1: "Phase 1\nMinimal", 2: "Phase 2\nStressed", 3: "Phase 3\nCrisis"}
    colours      = {1: "#66BB6A", 2: "#FFA726", 3: "#EF5350"}

    for phase, group in ipc_weather.groupby("ipc_phase_county"):
        ax.boxplot(group["spi_3"].dropna(),
                   positions=[phase],
                   patch_artist=True,
                   boxprops=dict(facecolor=colours.get(phase, "grey"), alpha=0.7))

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([phase_labels.get(p, str(p)) for p in [1, 2, 3]])
    ax.axhline(0, color="black", linestyle="--", alpha=0.5, label="Normal rainfall")
    ax.set_title("Rainfall Anomaly (SPI-3) by IPC Food Security Phase", fontsize=13, fontweight="bold")
    ax.set_ylabel("Standardised Precipitation Index (SPI-3)")
    ax.set_xlabel("IPC Phase")
    ax.legend()
    plt.tight_layout()
    plt.savefig("reports/figures/04_rainfall_vs_ipc.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("IPC phase not in master — check merge in notebook 02")

## Visualisation 5 — News Article Frequency Over Time

In [ ]:
section("VIZ 5 — Agricultural News Volume 2025–2026")

news["date"] = pd.to_datetime(news["date"], utc=True, errors="coerce").dt.tz_localize(None)
news_monthly = news.set_index("date").resample("ME").size().reset_index(name="article_count")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(news_monthly["date"], news_monthly["article_count"], color="#1B5E20", alpha=0.8, width=20)
ax.set_title("Kenya Agricultural News Article Volume (Monthly)", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Articles Scraped")
plt.tight_layout()
plt.savefig("reports/figures/05_news_volume.png", dpi=150, bbox_inches="tight")
plt.show()

## Correlation Analysis

In [ ]:
section("CORRELATION — Weather Features vs Target Variables")

numeric_cols = [c for c in master.select_dtypes(include=[np.number]).columns
                if c not in ["year", "month", "latitude", "longitude"]]

corr = master[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix — All Numeric Variables", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/06_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()